In [ ]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sklearn as skl
import seaborn as sns

from datetime import datetime, timedelta
import math, random, os
import itertools
import gc
from tqdm import tqdm

pd.set_option('display.max_rows', 1000) # show all rows with scrollbar, don't use if there are many rows.
pd.set_option('display.max_columns', None)
plt.rcParams['font.family']='SimHei'
plt.rcParams['axes.unicode_minus']=False
#from IPython.core.interactiveshell import InteractiveShell
#InteractiveShell.ast_node_interactivity = 'all'
%autosave 180

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD, IncrementalPCA
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans, MiniBatchKMeans, SpectralClustering,\
                            AffinityPropagation, Birch, MeanShift, OPTICS
from sklearn.neural_network import BernoulliRBM as RBM
from sklearn.mixture import GaussianMixture
import keras
from keras.models import Model
from keras.layers import Dense, Input
import umap
import hdbscan

import plotly.graph_objs as go
import plotly.offline
plotly.offline.init_notebook_mode()

In [ ]:
SMALL_SIZE = 16
MEDIUM_SIZE = 18
BIG_SIZE = 20
LARGE_SIZE = 24

params = {
    'figure.figsize': (12, 8),
    'font.size': MEDIUM_SIZE,
    'figure.titlesize': LARGE_SIZE,
    'axes.titlesize': BIG_SIZE,
    'axes.labelsize': BIG_SIZE,
    'xtick.labelsize': MEDIUM_SIZE,
    'ytick.labelsize': MEDIUM_SIZE,
    'legend.fontsize': BIG_SIZE,
}
plt.rcParams.update(params)

In [ ]:
def AdjustExcelColumnWidth(df, writer, sheetName, fm = None, maxLen = None):
    workSheet = writer.sheets[sheetName]
    for idx, col in enumerate(df):
        series = df[col]
        if maxLen is None:
            maxLen = max((
                series.astype(str).map(len).max(),
                len(str(series.name))
                )) + 1
        workSheet.set_column(idx, idx, maxLen, fm)

In [ ]:
def DF_Summary(df):
    return pd.DataFrame({'Data Type':df.dtypes, 'Missing Data':df.isnull().sum(), 'Unique Values':df.nunique(axis=0, dropna=False)})

In [ ]:
def diff_pd(df1, df2):
    """Identify differences between two pandas DataFrames"""
    assert (df1.columns == df2.columns).all(), \
        'DataFrame column names are different'
    if any(df1.dtypes != df2.dtypes):
        'Data Types are different, trying to convert'
        df2 = df2.astype(df1.dtypes)
    if df1.equals(df2):
        return None
    else:
        # need to account for np.nan == np.nan returning True
        diff_mask = (df1 != df2) & ~(df1.isnull() & df2.isnull())
        ne_stacked = diff_mask.stack()
        changed = ne_stacked[ne_stacked]
        changed.index.names = ['id', 'col']
        difference_locations = np.where(diff_mask)
        changed_from = df1.values[difference_locations]
        changed_to = df2.values[difference_locations]
        return pd.DataFrame({'from': changed_from, 'to': changed_to},
                            index=changed.index)

In [ ]:
def mkdir(path):
    if not os.path.exists(path):
        os.makedirs(path)

## 1. Data Analysis

In [ ]:
df1 = pd.read_csv('../shuju/干扰指标数据/小区汇总.csv', encoding='gbk', dtype=str)
df1.rename(columns={'DN':'dn'}, inplace=True)
df1['type'] = df1['pinduan']+','+df1['daikuan']+','+df1['xiaoquzhishi']\
              +','+df1['zizaibojiange']+','+df1['prbkuai']

In [ ]:
dft = []
folder = '../shuju/干扰指标数据/时域数据/'
for f in os.listdir(folder):
    dft.append(pd.read_csv(folder + f, parse_dates=['begin_time', 'end_time'],
               dtype={'r1022_001':np.float32}))
dft = pd.concat(dft).drop('rownu', axis=1)
dft['r1022_001'] = 10*np.log10(dft['r1022_001'])
dft['gNB'] = dft['dn'].apply(lambda x:x.split(',')[0])
dft['dn_time'] = dft['dn'] + ',' + dft['begin_time'].apply(lambda x:x.strftime('%Y-%m-%d %H:%M:%S'))

In [ ]:
df = []
folder = '../shuju/干扰指标数据/频域数据/'
data_columns = ['r1023_{:0>3}'.format(i) for i in range(1,274)]
data_columns_160 = ['r1023_{:0>3}'.format(i) for i in range(1,161)]
data_columns_162 = ['r1023_{:0>3}'.format(i) for i in range(1,163)]
for f in tqdm(os.listdir(folder)):
    df0 = pd.read_csv(folder + f, parse_dates=['begin_time', 'end_time'],
                      dtype=dict.fromkeys(data_columns, np.float32))
    df0[data_columns] = 10*np.log10(df0[data_columns])
    df.append(df0)
df = pd.concat(df).drop('rownu', axis=1)

In [ ]:
len(df)

In [ ]:
DF_Summary(dft)

In [ ]:
len(dft)

In [ ]:
#plt.hist(df[data_columns].values.flatten(), bins=200);

In [ ]:
#plt.hist(dft['r1022_001'], bins=200);

In [ ]:
df = df[~df['r1023_010'].isna()]

In [ ]:
df = df.drop_duplicates(['dn', 'begin_time'])

In [ ]:
dft = dft.drop_duplicates(['dn', 'begin_time'])

In [ ]:
df = df1.merge(df, on='dn')

In [ ]:
df = dft.merge(df, on=['dn','begin_time','end_time'])

In [ ]:
df = df[df['r1022_001'].values>=-115]

In [ ]:
df.loc[df['prbkuai']=='273',data_columns] = df.loc[df['prbkuai']=='273',data_columns]\
                                            .interpolate(axis=1,limit_direction='both')
df.loc[df['prbkuai']=='160',data_columns_160] = df.loc[df['prbkuai']=='160',data_columns_160]\
                                                .interpolate(axis=1,limit_direction='both')
df.loc[df['prbkuai']=='162',data_columns_162] = df.loc[df['prbkuai']=='162',data_columns_162]\
                                                .interpolate(axis=1,limit_direction='both')

In [ ]:
df = df.sort_values(['type','dn','begin_time'])

In [ ]:
pd.DataFrame(df['type'].value_counts())

In [ ]:
df.groupby('type').agg({'dn':lambda x:len(x.unique()), 'gNB':lambda x:len(x.unique())})

In [ ]:
len(df['gNB'].unique())

In [ ]:
# 聚类分类
np.random.seed(1234567)
n_clusters = 12
n_components = 50
folder = '专家打标'
mkdir(folder)
df_label = []
i = 1
for xiaoqu_type in df['type'].unique():
    print(xiaoqu_type)
    folder_path1 = folder + '/' + xiaoqu_type
    mkdir(folder_path1)
    
    Data_select = df[df['type'].values==xiaoqu_type]
    n_samples = min(1000000, len(Data_select))
    Data_select = Data_select.sample(n=n_samples)
    
    data_train = Data_select.loc[:, data_columns].fillna(0)
    scaler = StandardScaler()
    data_train = scaler.fit_transform(data_train)

    # Dimensionality Reduction
#     tsne = TSNE(n_components=n_components, method='exact', perplexity=100, learning_rate=100, n_iter=1000, n_iter_without_progress=500, verbose=2)
#     data_reduced = tsne.fit_transform(data_train)
#     tsvd = TruncatedSVD(n_components=n_components)
#     data_reduced = tsvd.fit_transform(data_train)

    data_reduced = data_train
    
    # Clustering
#     ac = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward').fit(data_reduced)
#     Data_select['Class'] = (ac.labels_+1).astype(str)
    
    km = KMeans(n_clusters=n_clusters, max_iter=1000).fit(data_reduced)
    Data_select['Class'] = (km.labels_+1).astype(str)

#     mb = MiniBatchKMeans(n_clusters=n_clusters, batch_size=2048, max_iter=1000).fit(data_reduced)
#     Data_select['Class'] = (mb.labels_+1).astype(str)
    
    if xiaoqu_type == '2496~2690MHz,100MHz,TDD室分站,30,273':
        data_select = Data_select.groupby('gNB').sample(n=1)
    elif xiaoqu_type == '2496~2690MHz,60MHz,TDD宏站,30,162':
        data_select = Data_select.groupby('dn').sample(n=10)
    elif xiaoqu_type == '758~803MHz,30MHz,FDD宏站,15,160':
        data_select = Data_select.groupby('gNB').sample(n=1).sample(n=100)
    else:
        data_select = Data_select.groupby('gNB').sample(n=1).sample(n=200)
    
    for c in data_select['Class'].unique():
        mkdir(folder_path1 + '/' + c)
    df_label.append(data_select[['dn_time', 'type']])
    
    for idx, row in tqdm(data_select.to_dict('index').items()):
        i = i + 1
        time1 = row['begin_time'] - timedelta(hours=12)
        time2 = row['begin_time'] + timedelta(hours=12)
        time_list = pd.date_range(time1, time2, freq='H')
        
        df0 = df[df['dn'].values==row['dn']].sort_values('begin_time')
        df0_select = df0[df0['begin_time'].isin(time_list)]
        
        dft0 = dft[dft['dn'].values==row['dn']].sort_values('begin_time')
        dft0_select = dft0[(dft0['begin_time']>=time1) & (dft0['begin_time']<=time2)]

        fig, axes = plt.subplots(2, 1)
        fig.set_size_inches(15,8)
        ax = axes[0]
        ax.plot(data_select.loc[idx, data_columns].index.map(lambda x:int(x[-3:])),
                df0_select.loc[:, data_columns].values.T)
        ax.axhline(y=-115, color='r', linestyle='--')
        ax.grid(axis='x')
        ax.set_xticks(range(0,290,10))
        ax.set_title('频域图' + str(i) + ': ' + row['dn_time'] + '的前后12小时内(不同颜色代表不同小时)')
        ax.set_xlabel('PRB')
        ax.set_ylabel('干扰值 (dBm)')
        
        ax = axes[1]
        ax.plot(dft0_select['begin_time'], dft0_select['r1022_001'])
        ax.axhline(y=-115, color='r', linestyle='--')
        ax.axvline(x=row['begin_time'], color='b', linestyle='--')
        ax.grid(axis='x')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H时'))
        plt.xticks(rotation=30, horizontalalignment='right')
        ax.set_title('时域参考图: ' + row['dn_time'] + '的前后12小时内')
        ax.set_xlabel('时间 (月-日 小时)')
        ax.set_ylabel('干扰值 (dBm)')
        fig.tight_layout()
        plt.savefig(folder_path1 + '/' + row['Class'] + '/频域图' + str(i) + '_' + row['dn_time'].replace(':','') + '.png')
        plt.close(fig)

In [ ]:
df_label = pd.concat(df_label)
df_label.columns = ['小区编号和干扰时间', '频段,带宽,小区制式,子载波间隔,PRB块数']
df_label['频域图号'] = ['频域图{}'.format(i) for i in range(2,len(df_label)+2)]
df_label['干扰类型(需要填充)'] = ''
df_label.to_excel('专家打标' + '/专家打标.xlsx', index=False)

In [ ]:
# df_select = df.groupby('dn').sample(n=1, random_state=1234567)

# for i in range(0,len(df_select),300):
#     fig, ax = plt.subplots()
#     fig.set_size_inches(10,3)
#     ax.plot(df_select[data_columns].values[i])
#     ax.set_title(df_select.iloc[i,0])
#     ax.set_xlabel('PRB')

# fig, ax = plt.subplots(figsize=(12, 10))
# data_plot = df_select.sample(n=1)
# data_plot['date'] = data_plot['begin_time'].apply(lambda x:x.strftime('%m-%d %H'))
# data_plot = data_plot.set_index('date')[data_columns]
# #plt.imshow(data_plot, aspect='auto');
# sns.heatmap(data_plot);

## 2. Dimensionality Reduction

In [ ]:
def plotly_cluster(data):
    labels = data['Label'].unique()
    scatters = []
    random.seed(1234567)
    for label in labels:
        color = '#'+''.join([random.choice(list('0123456789ABCDEF')) for j in range(6)])
        scatter = go.Scatter3d(
                            x=data[data['Label']==label]['X1'],
                            y=data[data['Label']==label]['X2'],
                            z=data[data['Label']==label]['X3'],
                            mode='markers',
                            marker=dict(
                                size=5,
                                color=color,
                                opacity=1
                            )
                        )
        layout = go.Layout(
                margin=dict(
                    l=0,
                    r=0,
                    b=0,
                    t=0
                ),
                font=dict(
                    size=data_ac_labeled.shape[1]-1
                ),
                showlegend=False
                #legend=dict(x=0.8, y=0.5, font=dict(size=16))
            )
        scatters.append(scatter)
    fig = go.Figure(data=scatters, layout=layout)
    #return ply.iplot(fig, filename='3d-scatter')
    plotly.offline.iplot(fig)

In [ ]:
df['type'].unique()

In [ ]:
np.random.seed(1234567)
data_train = df.loc[df['type'].values==df['type'].unique()[3], data_columns].sample(n=10000).fillna(0)
scaler = StandardScaler()
data_train = scaler.fit_transform(data_train)

In [ ]:
n_clusters = 10
n_components = 3
X_Columns = ['X' + str(i+1) for i in range(n_components)]

In [ ]:
pca = PCA(n_components=n_components)
data_pca = pca.fit_transform(data_train)
pca.explained_variance_ratio_
np.var((data_train - pca.inverse_transform(data_pca)))

In [ ]:
tsvd = TruncatedSVD(n_components=n_components)
data_tsvd = tsvd.fit_transform(data_train)
np.var((data_train - tsvd.inverse_transform(data_tsvd)))

In [ ]:
ipca = IncrementalPCA(n_components=n_components, batch_size=128)
data_ipca = ipca.fit_transform(data_train)
np.var((data_train - ipca.inverse_transform(data_ipca)))

In [ ]:
tsne = TSNE(n_components=n_components, method='exact', perplexity=100, learning_rate=100, n_iter=1000, n_iter_without_progress=500, verbose=2)
data_tsne = tsne.fit_transform(data_train)

In [ ]:
rbm = RBM(n_components=n_components, learning_rate=0.01, n_iter=100, batch_size=128)
data_rbm = rbm.fit_transform(data_train)

In [ ]:
umap_model = umap.UMAP(n_neighbors=100, min_dist=0.5, n_components=n_components, metric='minkowski')
data_umap = umap_model.fit_transform(data_train)

In [ ]:
import tensorflow as tf
tf.random.set_seed(1234567)

num_features = data_train.shape[1]
# input_data = Input(shape=(num_features,))

# # encoder layers
# encoded = Dense(40, activation='relu')(input_data)
# encoder_output = Dense(n_components)(encoded)

# # decoder layers
# decoded = Dense(40, activation='relu')(encoder_output)
# decoded = Dense(num_features, activation='tanh')(decoded)

# # construct the autoencoder model
# autoencoder = Model(inputs=input_data, outputs=decoded)

# # construct the encoder model for plotting
# encoder = Model(inputs=input_data, outputs=encoder_output)

encoder = keras.models.Sequential([
    keras.layers.Dense(50, activation='relu'),
    keras.layers.Dense(25, activation='relu'),
    keras.layers.Dense(n_components)
])

decoder = keras.models.Sequential([
    keras.layers.Dense(25, activation='relu'),
    keras.layers.Dense(50, activation='relu'),
    keras.layers.Dense(num_features, activation='tanh')
])

autoencoder = keras.models.Sequential([
    encoder,
    decoder
])

# compile autoencoder
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# training
autoencoder.fit(data_train, data_train, epochs=20, batch_size=128, shuffle=True)

In [ ]:
autoencoder.evaluate(data_train, data_train)

In [ ]:
data_encoder = encoder.predict(data_train)

## 3. Clustering

In [ ]:
data_reduced = data_tsvd

In [ ]:
ac = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward').fit(data_reduced)
data_ac_labeled = pd.concat([pd.Series(ac.labels_.astype(str), name='Label'),
                             pd.DataFrame(data_reduced, columns=X_Columns)], axis=1)
plotly_cluster(data_ac_labeled)

In [ ]:
db = DBSCAN(n_jobs=6).fit(data_reduced)
data_db_labeled = pd.concat([pd.Series(db.labels_.astype(str), name='Label'),
                             pd.DataFrame(data_reduced, columns=X_Columns)], axis=1)
plotly_cluster(data_db_labeled)

In [ ]:
hdb = hdbscan.HDBSCAN().fit(data_reduced)
data_hdb_labeled = pd.concat([pd.Series(hdb.labels_.astype(str), name='Label'),
                              pd.DataFrame(data_reduced, columns=X_Columns)], axis=1)
plotly_cluster(data_hdb_labeled)

In [ ]:
km = KMeans(n_clusters=n_clusters).fit(data_reduced)
data_km_labeled = pd.concat([pd.Series(km.labels_.astype(str), name='Label'),
                             pd.DataFrame(data_reduced, columns=X_Columns)], axis=1)
plotly_cluster(data_km_labeled)

In [ ]:
# sc = SpectralClustering(n_clusters=n_clusters, assign_labels='discretize').fit(data_reduced)
# data_sc_labeled = pd.concat([pd.Series(sc.labels_.astype(str), name='Label'),
#                              pd.DataFrame(data_reduced, columns=X_Columns)], axis=1)
# plotly_cluster(data_sc_labeled)

In [ ]:
# ap = AffinityPropagation(damping=0.9).fit(data_reduced)
# data_ap_labeled = pd.concat([pd.Series(ap.labels_.astype(str), name='Label'),
#                              pd.DataFrame(data_reduced, columns=X_Columns)], axis=1)
# plotly_cluster(data_ap_labeled)

In [ ]:
mb = MiniBatchKMeans(n_clusters=n_clusters, batch_size=2048).fit(data_reduced)
data_mb_labeled = pd.concat([pd.Series(mb.labels_.astype(str), name='Label'),
                             pd.DataFrame(data_reduced, columns=X_Columns)], axis=1)
plotly_cluster(data_mb_labeled)

In [ ]:
br = Birch(n_clusters=n_clusters).fit(data_reduced)
data_br_labeled = pd.concat([pd.Series(br.labels_.astype(str), name='Label'),
                             pd.DataFrame(data_reduced, columns=X_Columns)], axis=1)
plotly_cluster(data_br_labeled)

In [ ]:
ms = MeanShift().fit(data_reduced)
data_ms_labeled = pd.concat([pd.Series(ms.labels_.astype(str), name='Label'),
                             pd.DataFrame(data_reduced, columns=X_Columns)], axis=1)
plotly_cluster(data_ms_labeled)

In [ ]:
op = OPTICS().fit(data_reduced)
data_op_labeled = pd.concat([pd.Series(op.labels_.astype(str), name='Label'),
                             pd.DataFrame(data_reduced, columns=X_Columns)], axis=1)
plotly_cluster(data_op_labeled)

In [ ]:
gm = GaussianMixture(n_components=n_components).fit(data_reduced)
data_gm_labeled = pd.concat([pd.Series(gm.predict(data_reduced).astype(str), name='Label'),
                             pd.DataFrame(data_reduced, columns=X_Columns)], axis=1)
plotly_cluster(data_gm_labeled)